## Structured output

Models casn be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

## Pydantic

pydantic models provide the richest feature set with field validation, description and nested structures.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

In [15]:
from sqlalchemy import desc
# pydantic will help you build structured output

from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description='The title of the movie')
    year:int=Field(description='The year the movie was released')
    director:str=Field(description='The director of the movie')
    rating:float=Field(description='The movies rating out of 10')

In [16]:
model_with_structure = model.with_structured_output(Movie)
print(model_with_structure)

first=_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D8178B7620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D817A58440>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': '

In [17]:
response = model_with_structure.invoke('Provide details about the movie inception')
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output  alongside parsed structure

In [19]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        ...,
        description="The title of the movie"
    )

    year: int = Field(
        ...,
        description="The year the movie was released"
    )

    director: str = Field(
        ...,
        description="The director of the movie"
    )
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke('Provide details about the moview Inception')
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants details about the movie Inception. We have a function Movie that can provide details given director, title, year. We need to call the function with appropriate parameters: director: Christopher Nolan, title: Inception, year: 2010. Then we can output the result. Let's call the function.", 'tool_calls': [{'id': 'fc_4ddd843c-8194-48ea-bdfc-4b9471feb0d9', 'function': {'arguments': '{"director":"Christopher Nolan","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 147, 'total_tokens': 254, 'completion_time': 0.229567604, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.005728108, 'prompt_tokens_details': None, 'queue_time': 0.153453473, 'total_time': 0.235295712}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 't

In [ ]:
# 1:36